# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saadar2846/flyrank_ml_internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding A: "What Predicts Growth?" (ML Appendix — Logistic Regression, 71% holdout accuracy)**

*Where does the label come from?* The paper's `Trend Direction` field is derived from a 30-day vs.
prior-30-day impression change (Up: >10%, Down: >10%, Stable: within ±10%). This is a short, current-
window comparison, not a validated future outcome — a page sitting at +11% and one at +9% land in
different classes despite being nearly identical. Question: how sensitive is the 71% holdout accuracy
to pages sitting near that ±10% boundary? A brief note on how many pages fall within a few points of
the cutoff would help readers judge how sharp this "growing vs. declining" line really is.

*Does the validation design support the claim?* The methodology section states an 80/20 split for the
logistic regression but doesn't specify whether it's grouped by brand. With 57 brands in the portfolio,
pages from the same brand likely share structural traits (template, editorial process, niche) that a
model could partially memorize if brand pages appear on both sides of the split. Question: was the
split brand-grouped, and if not, would a grouped re-run change the 71% figure meaningfully?

---

**Finding B: "The Content Performance Curve" (Finding #2 — health score peaks at 61-90 days)**

*Where does the label come from?* Health score itself is a composite built partly from impressions and
position — the paper's own feature-importance appendix flags this same concern for a different model
("the target itself is partly constructed from some of these inputs"). For the age-curve finding, that
same caveat is worth restating explicitly here too, since the "lifecycle" story is built on health score.

*Does the validation design support the claim?* This is a cross-sectional comparison — different pages
at different ages, measured at one snapshot in time — not the same pages tracked longitudinally through
their own lifecycle. That distinction matters: if older pages that would have declined are more likely
to be pruned or unpublished, the 271-365 day bucket could be *survivorship-biased toward whatever old
content is still worth keeping live*, which would make the "decay cliff" look different from a true
per-page lifecycle curve. A respectful ask: it would strengthen this finding to note whether it's
cross-sectional or a tracked cohort, since the current phrasing ("content enters a growth phase...
hits peak performance") reads as if individual pages are being followed over time.

These aren't corrections — they're the same kind of question I'm about to turn on my own Week-5 work
in the sections below, in the same constructive spirit.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Re-running my Week-5 Random Forest under two splits on the same data: a naive random split (the
"before," representing what a less careful analysis might report) versus the client-grouped split I
already used in Week 5 (the "after," the honest version). Showing both Precision@k numbers side by side.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, subprocess
import pandas as pd, numpy as np

REPO_URL = "https://github.com/saadar2846/flyrank_ml_internship"
REPO_DIR = "flyrank_ml_internship"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count", "sessions_90d", "engagement_rate"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining_label"]

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# BEFORE: naive random split
X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
m_r = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42).fit(X_tr_r, y_tr_r)
score_r = m_r.predict_proba(X_te_r)[:, 1]

# AFTER: client-grouped split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, te_idx = next(gss.split(X, y, groups=df["client_id"]))
X_tr_g, X_te_g = X.iloc[tr_idx], X.iloc[te_idx]
y_tr_g, y_te_g = y.iloc[tr_idx], y.iloc[te_idx]
m_g = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42).fit(X_tr_g, y_tr_g)
score_g = m_g.predict_proba(X_te_g)[:, 1]

for k in (20, 50):
    print(f"Precision@{k}  random(before): {precision_at_k(score_r, y_te_r.values, k):.3f}  "
          f"|  grouped(after): {precision_at_k(score_g, y_te_g.values, k):.3f}")

Precision@20  random(before): 1.000  |  grouped(after): 0.850
Precision@50  random(before): 0.960  |  grouped(after): 0.680


**Before/after:** [state the real precision@20/50 numbers for both splits and whether the grouped
split came in lower, similar, or higher — report it honestly either way, since a small gap is itself
a valid, useful finding, not a failure to find one.]

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Re-running the Week-3 leakage checklist against my final Week-5 feature set:

- **Are any features calculated after the decision point?** No — `content_age_days`,
  `days_since_last_update`, `impressions_90d`, `avg_position`, `ctr`, `word_count`, `sessions_90d`,
  `engagement_rate` are all observable at prediction time.
- **Does the feature window overlap the target window?** The starter label (`trend_direction == down`)
  is a current-state bucket, not a distinct future window, so there's no explicit window overlap by
  construction — but this also means the label itself is closer to Finding A's boundary-sensitivity
  concern above than a clean future outcome would be. Noted as a limitation, not resolved here.
- **Did any product decision flag slip in as a feature?** No — `health_score`, `priority_score`,
  `action_type` aren't in this dataset.
- **Does a derived field secretly encode the target?** Checked each feature against `trend_direction`
  — none are computed from it.
- **Are related rows split unfairly across train/test?** Addressed directly in Section 2's
  before/after comparison.

Deliberate-leak demonstration (same trick as Week 2/3, confirmed again here):

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Add one label-derived column on purpose, watch the score jump, then remove it
X_leaky = X.copy()
if "trend_pct" in df.columns:
    X_leaky["trend_pct_LEAK"] = df["trend_pct"]
X_tr_l, X_te_l, y_tr_l, y_te_l = train_test_split(X_leaky, y, test_size=0.2, random_state=42, stratify=y)
m_l = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42).fit(X_tr_l, y_tr_l)
score_l = m_l.predict_proba(X_te_l)[:, 1]

from sklearn.metrics import average_precision_score
print(f"With leak — Average Precision: {average_precision_score(y_te_l, score_l):.3f}")
print(f"Without leak (honest, from Section 2) — Average Precision: "
      f"{average_precision_score(y_te_r, score_r):.3f}")

With leak — Average Precision: 1.000
Without leak (honest, from Section 2) — Average Precision: 0.763


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**My boldest sentence (Week 4):** "Staleness confirmed as a real signal behind decline."

**Rewritten in safe language:** "In this bucketed check (n reported per bucket in Week 4), pages
with longer `days_since_last_update` were observed to have a higher decline rate — a directional,
observational pattern, not a proven cause of decline. It's decision-support evidence for
prioritizing review, not a guarantee that refreshing a stale page will reverse its trend."

Going forward, all claims in this project use: observed, measured, directional, decision-support —
never "proven," "causes," or any claim about Google's ranking algorithm.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.